# Laboratorio 6 – Juegos Adversariales: Connect Four
**CC3045 – Inteligencia Artificial**

Este notebook implementa una IA para Connect Four (4 en línea) usando:
- **Task 2.1**: Lógica del juego + Agente Minimax puro
- **Task 2.2**: Optimización con Poda Alfa-Beta + comparación de nodos visitados
- **Task 2.3**: Función heurística de evaluación + demos

---
## Task 2.1 – Lógica del juego y Agente Minimax

In [ ]:
import numpy as np
import random
import time

class Connect4:
    ROWS = 6
    COLS = 7
    EMPTY  = 0
    PLAYER = 1   # humano / MIN
    AI     = 2   # agente  / MAX
    WIN_SCORE  =  1000
    LOSS_SCORE = -1000

    def __init__(self):
        # Tablero 6x7 inicializado en 0 (vacio)
        self.board = np.zeros((self.ROWS, self.COLS), dtype=int)

    def copy(self):
        """Copia profunda del estado — necesaria para la recursion de Minimax."""
        new_game = Connect4()
        new_game.board = self.board.copy()
        return new_game

    # actions(s): movimientos validos
    def get_valid_columns(self):
        """
        Equivale a actions(s) en la formulacion de Minimax.
        Una columna es valida si su celda superior sigue vacia.
        """
        return [c for c in range(self.COLS) if self.board[0][c] == self.EMPTY]

    def drop_piece(self, col, piece):
        """
        result(s, a): aplica la accion `col` al estado.
        La ficha cae por gravedad a la fila mas baja disponible.
        """
        for row in range(self.ROWS - 1, -1, -1):
            if self.board[row][col] == self.EMPTY:
                self.board[row][col] = piece
                return row
        return -1

    # is_terminal(s): deteccion de victoria
    def check_win(self, piece):
        """
        Verifica si `piece` conecto 4 fichas consecutivas
        en cualquier direccion: horizontal, vertical, diagonal.
        """
        b = self.board
        # Horizontal
        for r in range(self.ROWS):
            for c in range(self.COLS - 3):
                if all(b[r][c+i] == piece for i in range(4)):
                    return True
        # Vertical
        for r in range(self.ROWS - 3):
            for c in range(self.COLS):
                if all(b[r+i][c] == piece for i in range(4)):
                    return True
        # Diagonal ascendente
        for r in range(3, self.ROWS):
            for c in range(self.COLS - 3):
                if all(b[r-i][c+i] == piece for i in range(4)):
                    return True
        # Diagonal descendente
        for r in range(self.ROWS - 3):
            for c in range(self.COLS - 3):
                if all(b[r+i][c+i] == piece for i in range(4)):
                    return True
        return False

    def is_terminal(self):
        """is_terminal(s): alguien gano o el tablero esta lleno."""
        return (
            self.check_win(self.PLAYER) or
            self.check_win(self.AI) or
            len(self.get_valid_columns()) == 0
        )

    def print_board(self):
        symbols = {0: '.', 1: 'R', 2: 'Y'}
        print('  ' + '  '.join(str(c) for c in range(self.COLS)))
        for row in self.board:
            print('  ' + '  '.join(symbols[cell] for cell in row))
        print()

print('Task 2.1 - Clase Connect4 definida correctamente.')

Task 2.1 - Clase Connect4 definida correctamente.


In [6]:
# ══════════════════════════════════════════════════════════
# FUNCION HEURISTICA evaluate_board(board)
#
# NOTA IMPORTANTE: Esta funcion se define AQUI, antes de minimax(),
# porque tanto Minimax como Alfa-Beta la invocan cuando depth == 0.
# La explicacion detallada de la estrategia esta en la seccion Task 2.3.
#
# Estrategia:
#   1. Control del centro: la col central (3) conecta mas lineas posibles -> +3 por ficha
#   2. Ventanas de 4 celdas en las 4 direcciones:
#      +1000 si IA tiene 4     (victoria)
#       +50  si IA tiene 3 + 1 vacio   (amenaza inmediata)
#        +5  si IA tiene 2 + 2 vacios  (posicion en desarrollo)
#       -80  si rival tiene 3 + 1 vacio (bloquear amenaza)
# ══════════════════════════════════════════════════════════

def score_window(window, piece):
    """Evalua una ventana de 4 celdas consecutivas desde la perspectiva de `piece`."""
    opp   = Connect4.PLAYER if piece == Connect4.AI else Connect4.AI
    score = 0
    cp = window.count(piece)
    ce = window.count(Connect4.EMPTY)
    co = window.count(opp)

    if   cp == 4:              score += 1000  # victoria segura
    elif cp == 3 and ce == 1:  score +=   50  # 3-en-linea con hueco
    elif cp == 2 and ce == 2:  score +=    5  # posicion prometedora
    if   co == 3 and ce == 1:  score -=   80  # bloquear amenaza rival
    return score


def evaluate_board(board):
    """
    Eval(s): funcion heuristica completa del tablero.
    Positivo -> favorable para IA | Negativo -> favorable para humano.
    """
    ROWS  = Connect4.ROWS
    COLS  = Connect4.COLS
    piece = Connect4.AI
    score = 0

    # 1. Bonus por control de la columna central
    center_col = [int(board[r][COLS // 2]) for r in range(ROWS)]
    score += center_col.count(piece) * 3

    # 2. Ventanas horizontales
    for r in range(ROWS):
        for c in range(COLS - 3):
            score += score_window([board[r][c+i] for i in range(4)], piece)

    # 3. Ventanas verticales
    for c in range(COLS):
        for r in range(ROWS - 3):
            score += score_window([board[r+i][c] for i in range(4)], piece)

    # 4. Diagonales ascendentes
    for r in range(3, ROWS):
        for c in range(COLS - 3):
            score += score_window([board[r-i][c+i] for i in range(4)], piece)

    # 5. Diagonales descendentes
    for r in range(ROWS - 3):
        for c in range(COLS - 3):
            score += score_window([board[r+i][c+i] for i in range(4)], piece)

    return score

print('Funcion heuristica evaluate_board() definida.')
print('(Ver seccion Task 2.3 para la explicacion de la estrategia)')

Funcion heuristica evaluate_board() definida.
(Ver seccion Task 2.3 para la explicacion de la estrategia)


In [7]:
# ══════════════════════════════════════════════════════════
# AGENTE MINIMAX PURO  (Task 2.1)
# Complejidad: O(b^d) sin ninguna optimizacion.
# Restriccion del lab: usar d=3 o d=4.
# ══════════════════════════════════════════════════════════

nodes_minimax = 0  # contador global para medir nodos explorados

def minimax(game, depth, is_maximizing):
    """
    Algoritmo Minimax recursivo.

    Formula (diapositivas):
      minimax(s) = utility(s)                    si is_terminal(s)
      minimax(s) = max_a minimax(result(s,a))     si turno MAX
      minimax(s) = min_a minimax(result(s,a))     si turno MIN

    Args:
      game           -- estado actual (Connect4)
      depth          -- profundidad restante
      is_maximizing  -- True: turno MAX (IA) | False: turno MIN (humano)
    """
    global nodes_minimax
    nodes_minimax += 1

    # Caso base 1: estado terminal -> retornar utilidad exacta
    if game.check_win(Connect4.AI):        return Connect4.WIN_SCORE   # +1000
    if game.check_win(Connect4.PLAYER):    return Connect4.LOSS_SCORE  # -1000
    if len(game.get_valid_columns()) == 0: return 0                     # empate

    # Caso base 2: profundidad agotada -> usar heuristica Eval(s)
    if depth == 0:
        return evaluate_board(game.board)

    valid_cols = game.get_valid_columns()

    if is_maximizing:
        # MAX elige el movimiento de mayor valor
        best_value = float('-inf')
        for col in valid_cols:
            child = game.copy()
            child.drop_piece(col, Connect4.AI)        # result(s, a)
            value = minimax(child, depth - 1, False)
            best_value = max(best_value, value)
        return best_value
    else:
        # MIN elige el movimiento de menor valor
        best_value = float('inf')
        for col in valid_cols:
            child = game.copy()
            child.drop_piece(col, Connect4.PLAYER)    # result(s, a)
            value = minimax(child, depth - 1, True)
            best_value = min(best_value, value)
        return best_value


def get_best_move_minimax(game, depth=3):
    """
    Funcion principal del agente Minimax puro.
    Evalua cada columna valida y elige la de mayor valor (perspectiva MAX=IA).
    """
    global nodes_minimax
    nodes_minimax = 0
    best_col   = None
    best_value = float('-inf')
    for col in game.get_valid_columns():
        child = game.copy()
        child.drop_piece(col, Connect4.AI)
        value = minimax(child, depth - 1, False)
        if value > best_value:
            best_value = value
            best_col   = col
    return best_col

print('Task 2.1 - Agente Minimax puro listo. Usar d=3 o d=4.')

Task 2.1 - Agente Minimax puro listo. Usar d=3 o d=4.
